<!-- paper-paired-role-banner -->
> 🟩 **정답 코드 · 오른쪽 창** — 왼쪽과 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 구현 이유와 텐서 shape를 설명해 보세요.

# 04. Dropout — Dropout: A Simple Way to Prevent Neural Networks from Overfitting

- **원 논문:** [Dropout: A Simple Way to Prevent Neural Networks from Overfitting](https://jmlr.org/papers/volume15/srivastava14a/srivastava14a.pdf)
- **저자 / 발표:** Nitish Srivastava, Geoffrey Hinton, Alex Krizhevsky, Ilya Sutskever, and Ruslan Salakhutdinov · Journal of Machine Learning Research 15 (2014)
- **난이도 / 예상 시간:** 초중급 · 약 75분
- **선수 지식:** MLP, Bernoulli sampling, train/eval mode, expectation

이 노트북은 논문 결과 수치를 그대로 재현하는 대규모 benchmark가 아니라,
핵심 수식과 구조가 실제로 어떻게 동작하는지 확인하는 **하드웨어 인식 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하며, 작은 NumPy·Python 실험은
전송 비용을 피하기 위해 CPU에서 실행될 수 있습니다.

## 재현 범위와 완료 기준

**이 노트북이 재현하는 것**

논문 §2의 retain probability p를 사용하는 Bernoulli mask와 test-time weight scaling을 직접 구현한다. train/test 출력의 기대값 일치, masking 비율, 모듈 모드 전환을 검증하고 작은 XOR-like 분류기에서 loss 및 decision boundary를 관찰한다.

**원 논문과 다른 것**

원 논문은 MNIST, TIMIT, CIFAR-10/100, SVHN, ImageNet에서 큰 networks와 여러 hyperparameter를 비교한다. 여기서는 80개의 2D Gaussian points와 2-layer MLP만 사용해 stochastic thinning과 test-time scaling 메커니즘을 검증한다.

완료하려면 TODO를 구현하고, 검증 셀의 `assert`를 모두 통과시키고, 마지막 관찰 질문에
자신의 말로 답하세요. 수치가 논문과 다르더라도 핵심 불변식과 비교 방향이 맞으면 성공입니다.

## 논문 ↔ 노트북 지도

        절 번호와 수식 번호는 위 링크의 대표 공개본을 기준으로 합니다. 판본에 따라 페이지는 달라질 수
        있으므로 **절 제목과 수식 번호**를 함께 사용하세요.

        | 원 논문의 위치 | 이 노트북의 대응 실습 | 확인할 증거 |
        |---|---|---|
        | §2, Fig. 1–2: 각 hidden unit을 probability p로 retain | `paper_dropout`의 Bernoulli mask | 10,000개 unit의 실제 retain fraction이 p 근처인지 assert |
| §2, Eq. (1): r_j^(l) ~ Bernoulli(p), y~=r*y | training branch의 `mask * x` | 고정 generator로 mask와 출력의 재현성 확인 |
| §2 test time: outgoing weights에 p를 곱한 scaled network | activation 관점에서 같은 `p * x`를 반환하는 eval branch | train Monte-Carlo mean과 deterministic eval mean 비교 |
| §7.2 ‘Effect of Dropout on Sparsity’: sparse activations 관찰 | `DropoutMLP` hidden activation mask와 합성 분류 실험 | loss curve와 2D decision probability map을 표시 |

        > 읽기 순서: 먼저 해당 절을 훑고 → 코드를 예측해 작성하고 → 출력이 논문의 주장과 왜
        > 연결되는지 한 문장으로 설명합니다.

## 핵심 수식과 구현 설계

$$
m_i\sim\operatorname{Bernoulli}(q),\qquad
h_i^{\mathrm{train}}=m_i h_i,\qquad
h_i^{\mathrm{test}}=q h_i
$$

- $q=1-p$는 unit 보존 확률이고 $m_i$는 학습 시 샘플되는 mask입니다.
- 위 식은 원 논문의 관례입니다. train에서는 mask만 곱하고 test에서는 outgoing weight를
  $q$배하며, 실습은 동치인 activation $q h_i$를 계산합니다.
- PyTorch의 inverted dropout은 train에서 $m_i h_i/q$, eval에서 $h_i$를 사용합니다.
  두 관례의 기대값은 같지만 각 mode의 중간 activation scale은 다릅니다.
- 실습은 원 논문 관례의 mask, mode 전환, PyTorch 관례와의 차이를 함수로 확인합니다.
- activation shape은 유지되지만 train mode에서만 원소별 0이 생깁니다.

## 구현 선택 이유

- **원문에서 보존한 부분:** §2, Fig. 1–2: 각 hidden unit을 probability p로 retain
- **노트북 구현:** `paper_dropout`의 Bernoulli mask
- **이렇게 구현한 이유:** 완성된 고수준 model을 한 줄로 호출하면
  논문의 핵심 shape·mask·loss·상태 변화가 숨겨집니다. 따라서 이 계산을
  이름이 드러나는 class와 함수로 나누고 중간 tensor를 assertion으로
  검사합니다.
- **실패를 잡는 증거:** 10,000개 unit의 실제 retain fraction이 p 근처인지 assert를 출력 크기만 아니라
  구현 불변식으로 사용합니다.
- **축소 선택:** 원 논문은 MNIST, TIMIT, CIFAR-10/100, SVHN, ImageNet에서 큰 networks와 여러 hyperparameter를 비교한다. 여기서는 80개의 2D Gaussian points와 2-layer MLP만 사용해 stochastic thinning과 test-time scaling 메커니즘을 검증한다.는 기본 실행에서 생략합니다.
  이는 Windows CPU에서도 반복 실험이 가능하게 하면서 핵심 메커니즘은
  유지하기 위한 선택입니다.

## 1. 논문식 dropout과 PyTorch식 dropout

논문은 training 때 `mask*x`를 쓰고 test 때 weights를 `p`배 합니다. 이 실습은 activation
관점에서 test 출력을 `p*x`로 계산합니다. PyTorch의 `nn.Dropout(dropout_probability)`은
**inverted dropout**이라 training 때 살아남은 값을 `1/p`배하고 eval에서는 그대로 둡니다.
두 관례의 기대값은 같지만 중간 tensor의 scale이 다릅니다. 여기서 `p`는 drop 확률이
아니라 **retain 확률**임에 주의하세요.

### 구현 단계 1 · 실행 환경·데이터 계약 (setup 예외)

- **원문 위치:** 해당 없음 — import, seed, device, 로컬 데이터 로딩만 담당하며 논문의 학습 연산을 구현하지 않습니다.
- **Tensor shape 계약:** activation [B,H] → 같은 shape의 masked/scaled activation [B,H]
- **구현 이유:** 이후 셀의 비교가 재현 가능하도록 장치·dtype·seed와 입력 불변식을 먼저 고정합니다.
- **완료 증거:** 데이터 존재 여부, 입력 rank, 장치 선택 assertion과 환경 요약 출력이 통과해야 합니다.

In [ ]:
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(4)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device


def make_xor_blobs(points_per_blob=20):
    generator = torch.Generator().manual_seed(505)
    centers = torch.tensor([[-1.0, -1.0], [-1.0, 1.0], [1.0, -1.0], [1.0, 1.0]])
    labels = torch.tensor([0, 1, 1, 0])
    points, targets = [], []
    for center, label in zip(centers, labels):
        noise = 0.22 * torch.randn(points_per_blob, 2, generator=generator)
        points.append(center + noise)
        targets.append(label.repeat(points_per_blob))
    return torch.cat(points), torch.cat(targets)


drop_x, drop_y = make_xor_blobs()
drop_x, drop_y = ACCELERATOR.move(drop_x, drop_y)
assert drop_x.shape == (80, 2)
assert drop_y.bincount().tolist() == [40, 40]
print(ACCELERATOR.summary())
print("XOR-like batch:", tuple(drop_x.shape))

### 구현 단계 2 · 핵심 연산·모델

- **원문 위치:** §2, Fig. 1–2: 각 hidden unit을 probability p로 retain / §2, Eq. (1): r_j^(l) ~ Bernoulli(p), y~=r*y / §2 test time: outgoing weights에 p를 곱한 scaled network
- **이 셀의 책임:** `paper_dropout`의 Bernoulli mask / training branch의 `mask * x` / activation 관점에서 같은 `p * x`를 반환하는 eval branch
- **Tensor shape 계약:** activation [B,H] → 같은 shape의 masked/scaled activation [B,H]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** 10,000개 unit의 실제 retain fraction이 p 근처인지 assert / 고정 generator로 mask와 출력의 재현성 확인 / train Monte-Carlo mean과 deterministic eval mean 비교. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def paper_dropout(x, retain_probability=0.5, training=True, generator=None):
    if not 0.0 < retain_probability <= 1.0:
        raise ValueError("retain_probability must be in (0, 1]")
    if not training:
        return retain_probability * x
    probabilities = torch.full_like(x, retain_probability)
    mask = torch.bernoulli(probabilities, generator=generator)
    return mask * x


ones = torch.ones(10_000)
dropout_generator = torch.Generator().manual_seed(606)
sampled = paper_dropout(ones, 0.6, True, dropout_generator)
evaluated = paper_dropout(ones, 0.6, False)
retain_fraction = sampled.mean().item()
assert abs(retain_fraction - 0.6) < 0.02
assert torch.allclose(evaluated, torch.full_like(ones, 0.6))
assert abs(sampled.mean().item() - evaluated.mean().item()) < 0.02
print(
    f"sampled retain={retain_fraction:.3f}; "
    f"deterministic test mean={evaluated.mean():.3f}"
)

### 구현 단계 3 · 핵심 연산·모델

- **원문 위치:** §2, Fig. 1–2: 각 hidden unit을 probability p로 retain / §2, Eq. (1): r_j^(l) ~ Bernoulli(p), y~=r*y / §2 test time: outgoing weights에 p를 곱한 scaled network / §7.2 ‘Effect of Dropout on Sparsity’: sparse activations 관찰
- **이 셀의 책임:** `paper_dropout`의 Bernoulli mask / training branch의 `mask * x` / activation 관점에서 같은 `p * x`를 반환하는 eval branch / `DropoutMLP` hidden activation mask와 합성 분류 실험
- **Tensor shape 계약:** activation [B,H] → 같은 shape의 masked/scaled activation [B,H]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** 10,000개 unit의 실제 retain fraction이 p 근처인지 assert / 고정 generator로 mask와 출력의 재현성 확인 / train Monte-Carlo mean과 deterministic eval mean 비교 / loss curve와 2D decision probability map을 표시. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
class PaperDropout(nn.Module):
    def __init__(self, retain_probability=0.5):
        super().__init__()
        if not 0.0 < retain_probability <= 1.0:
            raise ValueError("retain_probability must be in (0, 1]")
        self.retain_probability = retain_probability

    def forward(self, x):
        return paper_dropout(x, self.retain_probability, self.training)


class DropoutMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(2, 24)
        self.fc2 = nn.Linear(24, 24)
        self.out = nn.Linear(24, 2)
        self.dropout = PaperDropout(retain_probability=0.75)

    def forward(self, x):
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.dropout(F.relu(self.fc2(x)))
        return self.out(x)


torch.manual_seed(4)
dropout_model = DropoutMLP().to(DEVICE)
dropout_model.eval()
first_eval = dropout_model(drop_x[:4])
second_eval = dropout_model(drop_x[:4])
assert torch.equal(first_eval, second_eval)
assert first_eval.shape == (4, 2)

### 구현 단계 4 · 평가·완료 증거

- **원문 위치:** §7.2 ‘Effect of Dropout on Sparsity’: sparse activations 관찰
- **이 셀의 책임:** `DropoutMLP` hidden activation mask와 합성 분류 실험
- **Tensor shape 계약:** activation [B,H] → 같은 shape의 masked/scaled activation [B,H]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** loss curve와 2D decision probability map을 표시. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
dropout_optimizer = torch.optim.Adam(dropout_model.parameters(), lr=0.03)
dropout_losses = []
dropout_model.train()
for _ in range(60):
    dropout_optimizer.zero_grad()
    loss = F.cross_entropy(dropout_model(drop_x), drop_y)
    loss.backward()
    dropout_optimizer.step()
    dropout_losses.append(float(loss.detach()))

dropout_model.eval()
with torch.no_grad():
    dropout_accuracy = (dropout_model(drop_x).argmax(1) == drop_y).float().mean().item()
assert torch.isfinite(torch.tensor(dropout_losses)).all()
assert min(dropout_losses[1:]) < dropout_losses[0]
print(
    f"loss {dropout_losses[0]:.3f} → {dropout_losses[-1]:.3f}; "
    f"accuracy={dropout_accuracy:.1%}"
)

### 구현 단계 5 · 평가·완료 증거

- **원문 위치:** §7.2 ‘Effect of Dropout on Sparsity’: sparse activations 관찰
- **이 셀의 책임:** `DropoutMLP` hidden activation mask와 합성 분류 실험
- **Tensor shape 계약:** activation [B,H] → 같은 shape의 masked/scaled activation [B,H]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** loss curve와 2D decision probability map을 표시. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
grid_axis = torch.linspace(-1.7, 1.7, 80, device=DEVICE)
grid_y, grid_x = torch.meshgrid(grid_axis, grid_axis, indexing="ij")
grid_points = torch.stack((grid_x.flatten(), grid_y.flatten()), dim=1)
dropout_model.eval()
with torch.no_grad():
    class_one_probability = dropout_model(grid_points).softmax(1)[:, 1].reshape(80, 80)

fig, axes = plt.subplots(1, 2, figsize=(9, 3.4))
image = axes[0].contourf(
    grid_x.detach().cpu(),
    grid_y.detach().cpu(),
    class_one_probability.detach().cpu(),
    levels=20,
    cmap="coolwarm",
)
axes[0].scatter(
    drop_x[:, 0].detach().cpu(),
    drop_x[:, 1].detach().cpu(),
    c=drop_y.detach().cpu(),
    cmap="coolwarm",
    edgecolor="black",
    s=18,
)
axes[0].set_title("test-time scaled decision map")
fig.colorbar(image, ax=axes[0], fraction=0.046)
axes[1].plot(dropout_losses)
axes[1].set_title("stochastic training loss")
axes[1].set_xlabel("optimization step")
axes[1].set_ylabel("cross-entropy")
axes[1].grid(alpha=0.3)
fig.tight_layout()
plt.show()
print(
    "decision probability range:",
    float(class_one_probability.min()),
    float(class_one_probability.max()),
)

## 마무리 해설과 다음 실험

1. 핵심 연산과 구현 위치를 원 논문의 용어로 다시 연결해 설명하세요.
2. 이 미니 실험은 `원 논문은 MNIST, TIMIT, CIFAR-10/100, SVHN, ImageNet에서 큰 networks와 여러 hyperparameter를 비교한다. 여기서는 80개의 2D Gaussian points와 2-layer MLP만 사용해 stochastic thinning과 test-time scaling 메커니즘을 검증한다.`를 재현하지 않으므로 논문의 최종 성능을 검증한 것이 아닙니다.
3. 다음에는 데이터 크기, 모델 폭, 학습 횟수 중 **하나만** 바꾸고 같은 seed에서 metric을 비교하세요.

정답의 수치 자체보다, shape·확률 정규화·mask·gradient 같은 불변식을 먼저 확인하는 습관이
논문 구현에서 가장 강한 디버깅 도구입니다.